# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?

Ans: Africa. Its population is about 3.9 times its 1970 size, and every other continent is far behind. Oceania (×2.3), South America (×2.3) and Asia (×2.2) are close together, North America is ×1.9, and Europe is barely up at ×1.1.


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Ans: **Line chart** of population **indexed to 1970 = 100**. The question is about *growth relative to 1970 size* over time, so a line chart (time on x) is the natural fit, and indexing puts a 5-billion-person continent and a 40-million-person one on the same scale. Africa is the single accent colour (it grows ×3.9, the fastest); the other five continents are grey context. Lines are labelled directly at their end points (no legend), gridlines and the x-axis title are removed, and the title states the finding.

In [ ]:
cont = (df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
               .sort_values(['Continent', 'Year']))
cont['Index'] = cont.groupby('Continent')['Population'].transform(lambda s: s / s.iloc[0] * 100)

ACCENT, GREY = '#0072B2', '#B0B0B0'
cmap = {c: (ACCENT if c == 'Africa' else GREY) for c in cont['Continent'].unique()}

fig = px.line(cont, x='Year', y='Index', color='Continent', color_discrete_map=cmap,
              template='plotly_white',
              title="<b>Africa's population has nearly quadrupled since 1970</b> — growing faster than any other continent")
fig.update_traces(line_width=2)
fig.update_traces(selector=dict(name='Africa'), line_width=4)
last = cont[cont['Year'] == 2022].sort_values('Index', ascending=False)
ypos, prev = [], None
for v in last['Index']:
    y = v if prev is None else min(v, prev - 20)
    ypos.append(y); prev = y
for (_, r), y in zip(last.iterrows(), ypos):
    fig.add_annotation(x=2022, y=y, xref='x', yref='y', xanchor='left', xshift=8, showarrow=False,
                       text=f"{r['Continent']}  ×{r['Index']/100:.1f}",
                       font=dict(color=ACCENT if r['Continent'] == 'Africa' else '#6b6b6b',
                                 size=14 if r['Continent'] == 'Africa' else 12))

fig.add_hline(y=100, line_dash='dot', line_color='#888')
fig.update_layout(showlegend=False, margin=dict(r=170), height=520)
fig.update_xaxes(title=None, showgrid=False, tickvals=[1970, 1980, 1990, 2000, 2010, 2022])
fig.update_yaxes(title='Population index (1970 = 100)', showgrid=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

Ans: India added about 860M people and China about 603M. Next come Pakistan (177M), Nigeria (163M), Indonesia (160M), the United States (138M), Brazil (119M), Bangladesh (104M), Ethiopia (95M), DR Congo (79M), the Philippines (78M), Mexico (77M), Egypt (76M), Iran (60M) and Vietnam (56M). Four of the 15 are in Africa: Nigeria, Ethiopia, DR Congo and Egypt. Asia has eight.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Ans: **Sorted horizontal bar chart.** The question is a ranking of 15 named countries by one quantity (people added), so bars on a common zero baseline sorted from largest to smallest are the most accurate encoding, and horizontal bars keep the country names readable. Colour carries the second part of the question: Africa is the accent colour, every other continent is grey, so the Africa count is visible at a glance. Values are printed at the bar ends, which lets me drop the x-axis and gridlines.

In [ ]:
d = df.assign(Added=(df['2022 Population'] - df['1970 Population']) / 1e6)
d = d.nlargest(15, 'Added').sort_values('Added')
d['Region'] = np.where(d['Continent'] == 'Africa', 'Africa', 'Other continents')
n_af = int((d['Continent'] == 'Africa').sum())

fig = px.bar(d, x='Added', y='Country', orientation='h', color='Region',
             color_discrete_map={'Africa': '#0072B2', 'Other continents': '#B0B0B0'},
             template='plotly_white', text='Added',
             title=f"<b>{n_af} of the 15 countries that added the most people since 1970 are in Africa</b> — Asia still dominates the top")
fig.update_traces(texttemplate='%{x:.0f}M', textposition='outside', cliponaxis=False)
fig.update_layout(legend=dict(title=None, orientation='h', y=1.02, x=0, yanchor='bottom'),
                  margin=dict(r=60), height=600, bargap=0.25)
fig.update_xaxes(visible=False)
fig.update_yaxes(title=None)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

Only loosely. The cloud rises from left to right, so larger countries tend to have more people, but the scatter is wide and area explains little of the difference. By continent, Asia and Europe sit higher (more people for their size), while Oceania and North America sit lower. The countries that break the pattern are:

1. Far above the trend: China, India and Bangladesh, plus the small city-states (Monaco, Macau, Singapore, Hong Kong).
2. Far below the trend: Russia, Canada, Australia and especially Greenland, which are huge but sparsely populated.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Ans:**Scatter plot** (area on x, 2022 population on y, one point per country). The question asks about the *relationship between two numeric variables*, which is exactly what a scatter shows; colour encodes continent so the continent differences are visible. Both axes are logarithmic because area and population span many orders of magnitude (a linear scale would squash almost all countries into the corner). The cloud is very loose, so land area explains little of population. The clear outliers (China and India far above the cloud, Russia, Canada, Australia and Greenland far to the right but sparsely populated, and city-states like Macau, Monaco, Singapore and Hong Kong squeezed far left) are labelled directly. Okabe-Ito colours keep the palette colour-blind safe.

In [12]:
okabe = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#56B4E9',
         'North America': '#009E73', 'Oceania': '#CC79A7', 'South America': '#D55E00'}

fig = px.scatter(df, x='Area', y='2022 Population', color='Continent', color_discrete_map=okabe,
                 hover_name='Country', log_x=True, log_y=True, template='plotly_white',
                 category_orders={'Continent': list(okabe)},
                 labels={'Area': 'Land area (km², log scale)', '2022 Population': '2022 population (log scale)'},
                 title="<b>Land area barely predicts population</b> — China and India dwarf everyone; Russia, Canada and Greenland are huge but sparse")
fig.update_traces(marker=dict(size=9, opacity=0.75, line=dict(width=0.5, color='white')))

outliers = ['China', 'India', 'Russia', 'Canada', 'Australia', 'Greenland',
            'Macau', 'Monaco', 'Singapore', 'Hong Kong', 'Bangladesh', 'United States']
for _, r in df[df['Country'].isin(outliers)].iterrows():
    fig.add_annotation(x=np.log10(r['Area']), y=np.log10(r['2022 Population']),   # log axes -> log10 coords
                       text=r['Country'], showarrow=False, xanchor='left', xshift=8,
                       font=dict(size=11, color='#444'))

fig.update_layout(legend=dict(title=None, orientation='h', y=1.02, x=0, yanchor='bottom'), height=620)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

Ans: Density varies much more within a continent than between continents. Median density is highest in North America, Asia and Europe (roughly 100–200 people/km²), lower in Oceania and Africa, and lowest in South America (under 20). Every continent except South America spans several orders of magnitude, so the averages hide the extremes. The most crowded places are Asian and European city-states (Macau, Singapore, Hong Kong, Monaco).

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Ans: **Box plot with every country shown as a point**, one box per continent. The question asks for the *distribution* of density within and across groups, and "spread, not just the average", which is what a box plot (median, quartiles, whiskers, outliers) is built for, and it lets me compare many groups side by side. Overlaying the individual points shows the real number of countries and the extremes. A log y-axis is needed because density runs from under 1 to over 20,000 people/km². Colour is a single neutral grey because the x-axis already identifies the continent; continents are sorted by median density, and the most extreme countries are labelled.

In [13]:
order = df.groupby('Continent')['Density'].median().sort_values(ascending=False).index.tolist()

fig = px.box(df, x='Continent', y='Density', points='all', hover_name='Country', log_y=True,
             category_orders={'Continent': order}, color_discrete_sequence=['#7F8FA6'],
             template='plotly_white',
             labels={'Density': 'People per km² (log scale)'},
             title="<b>Density varies far more within continents than between them</b> — the most crowded places are Asian and European city-states")
fig.update_traces(marker=dict(size=5, opacity=0.55), jitter=0.4, pointpos=0, boxmean=False)

yshifts = {'Macau': 0, 'Monaco': 0, 'Singapore': 8, 'Hong Kong': -8}   # separate the two close labels
for name, ys in yshifts.items():
    r = df[df['Country'] == name].iloc[0]
    fig.add_annotation(x=r['Continent'], y=np.log10(r['Density']), text=name, showarrow=False,
                       xanchor='left', xshift=14, yshift=ys, font=dict(size=11, color='#444'))

fig.update_layout(showlegend=False, height=600)
fig.update_xaxes(title=None, showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

Ans: 34 countries and territories shrank, mostly in Eastern Europe (the Baltics, Ukraine and the Balkans), with a few small island territories too. Russia is roughly flat. Growth is strongest in sub-Saharan Africa (many countries more than doubled), then the Middle East and Central Asia. The Americas and Australia grew moderately.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Ans:  **Choropleth map** — the question is explicitly about *where in the world*, so geography is the key variable and a map is the right chart. Colour shows percentage change 2000 → 2022 on a **diverging** scale centred exactly on 0% (orange = shrank, purple = grew, white = no change), which is colour-blind safe (no red/green). The scale is clipped at +160% so a few extreme growers (e.g. Gulf states) do not wash out the rest; the colour bar says so. Gridlines, frame and axes are removed, and the title states the finding.

In [ ]:
d = df.copy()
d['Change'] = (d['2022 Population'] / d['2000 Population'] - 1) * 100
n_shrank = int((d['Change'] < 0).sum())

lo, hi = -40, 160
zero = (0 - lo) / (hi - lo)
scale = [[0, '#B35806'], [zero, '#F7F7F7'], [1, '#542788']]
fig = px.choropleth(d, locations='CCA3', locationmode='ISO-3', color='Change', hover_name='Country',
                    color_continuous_scale=scale, range_color=[lo, hi], template='plotly_white',
                    labels={'Change': '% change'},
                    title=f"<b>{n_shrank} countries and territories shrank between 2000 and 2022</b> — mostly in Eastern Europe — while Africa and the Middle East grew fastest")
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_layout(coloraxis_colorbar=dict(title='% change<br>2000–2022<br>(clipped at 160)', thickness=14, len=0.6),
                  margin=dict(l=0, r=0, b=0), height=560)
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

Ans: Asia holds about 59% of the world, and China and India alone are over a third (about 18% each). Africa has about 18% (led by Nigeria and Ethiopia). Europe has about 9%, North and South America together about 13%, and Oceania under 1%.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Ans: **Treemap.** The data is a *part-to-whole hierarchy* (world → continent → country), and the question asks for both levels in one figure. A treemap nests countries inside their continent, with rectangle area proportional to 2022 population, so the share of every continent and every country is visible at once (a pie chart or stacked bar would only show one level). Colour identifies the continent (meaningful grouping, Okabe-Ito colour-blind-safe palette), and labels show the percentage of the world total.

In [ ]:

okabe = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#56B4E9',
         'North America': '#009E73', 'Oceania': '#CC79A7', 'South America': '#D55E00', '(?)': '#FFFFFF'}

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map=okabe, template='plotly_white',
                 title="<b>Asia holds about 59% of humanity</b> — China and India alone account for over a third of the world")
fig.update_traces(textinfo='label+percent root', root_color='white',
                  marker=dict(line=dict(width=1, color='white')),
                  hovertemplate='<b>%{label}</b><br>Population: %{value:,}<br>Share of world: %{percentRoot:.1%}<extra></extra>')
fig.update_layout(margin=dict(l=10, r=10, b=10), height=650)
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

Ans: Africa’s share rose from about 10% to 18%. Europe’s fell from about 18% to 9%. Asia’s is roughly stable (58% to 59%, peaking near 61% around 2000), and the rest of the world stays at about 14%.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Ans: **100% stacked bar chart** (one bar per year). The question is about *shares of a whole* (they must sum to 100%) and how those shares change over time, so a 100% stacked bar is the right choice: each bar is the whole world, and the segments show the composition. Africa sits at the bottom on a common baseline so its growth is easy to read; Africa (blue) and Europe (orange) are the two story colours, Asia is a light blue and the remainder is grey. Segments are labelled with their percentages, groups are named directly at the right of the last bar (no legend), and the title states the finding.

In [11]:
d = df_long.copy()
d['Group'] = np.where(d['Continent'].isin(['Africa', 'Asia', 'Europe']), d['Continent'], 'Rest of the world')
g = d.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
g['Share'] = g['Population'] / g.groupby('Year')['Population'].transform('sum') * 100
g['Year'] = g['Year'].astype(str)

stack_order = ['Africa', 'Asia', 'Europe', 'Rest of the world']
colors = {'Africa': '#0072B2', 'Asia': '#9ECAE1', 'Europe': '#E69F00', 'Rest of the world': '#D9D9D9'}
s = g.pivot(index='Year', columns='Group', values='Share')

fig = px.bar(g, x='Year', y='Share', color='Group', color_discrete_map=colors, template='plotly_white',
             category_orders={'Group': stack_order, 'Year': sorted(g['Year'].unique())},
             text=g['Share'].round(0).astype(int).astype(str) + '%',
             title=f"<b>Africa's share of world population rose from {s.loc['1970','Africa']:.0f}% to {s.loc['2022','Africa']:.0f}%</b> while Europe's fell from {s.loc['1970','Europe']:.0f}% to {s.loc['2022','Europe']:.0f}%")
fig.update_traces(textposition='inside', insidetextanchor='middle', textfont=dict(size=12, color='#222'))
fig.update_traces(selector=dict(name='Africa'), textfont=dict(color='white'))

cum = 0
for grp in stack_order:
    v = s.loc['2022', grp]
    fig.add_annotation(xref='paper', x=1.0, xanchor='left', xshift=8, y=cum + v / 2, yref='y',
                       text=grp, showarrow=False, font=dict(size=13, color='#444'))
    cum += v

fig.update_layout(showlegend=False, barmode='stack', bargap=0.2, margin=dict(r=140), height=560)
fig.update_xaxes(title=None, type='category')
fig.update_yaxes(title='Share of world population (%)', showgrid=False, range=[0, 100])
fig.show()